In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║   SCRIPT 2 (v3 FIXED) — TWO-STAGE TEMPORAL FUSION TRANSFORMER       ║
# ║   Fix: VSN rewritten using Keras layers only — no raw tf.ops        ║
# ║   Hardware: Google Colab T4 GPU — FRESH SESSION                     ║
# ║   Run each cell top to bottom                                        ║
# ╚══════════════════════════════════════════════════════════════════════╝


# ─────────────────────────────────────────────────────────────────
# CELL 1 — Mount Drive & Imports
# ─────────────────────────────────────────────────────────────────
from google.colab import drive
import os, gzip, io
import numpy as np
import joblib
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D,
    Add, Multiply, Flatten, Activation,
    LSTM, Lambda, Concatenate
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    classification_report, confusion_matrix
)

print("Mounting Google Drive...")
drive.mount('/content/drive')

PROJECT_FOLDER = 'your_path'
os.chdir(PROJECT_FOLDER)
print(f"Working directory: {os.getcwd()}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU detected: {gpus[0].name}")
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print("WARNING: No GPU — training will be slow")

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)


# ─────────────────────────────────────────────────────────────────
# CELL 2 — Configuration
# ─────────────────────────────────────────────────────────────────
X_PATH        = "X_sequences.npy"
Y_PATH        = "y_targets.npy"
TARGET_SCALER = "target_scaler.gz"
TEST_INDICES  = "test_indices.npy"

TFT_S1_PATH   = "tft_stage1_classifier.keras"
TFT_S2_PATH   = "tft_stage2_regressor.keras"

TIME_STEPS    = 48
N_FEATURES    = 13
D_MODEL       = 128
N_HEADS       = 8
FF_DIM        = 256
N_BLOCKS      = 4
DROPOUT_RATE  = 0.2
BATCH_SIZE    = 256
EPOCHS        = 100
VAL_SPLIT     = 0.15
TEST_SPLIT    = 0.10


# ─────────────────────────────────────────────────────────────────
# CELL 3 — Load Data & Create Two-Stage Labels
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  STEP 1: Loading Data")
print("="*55)

def load_scaler(path):
    with gzip.open(path, 'rb') as f:
        return joblib.load(io.BytesIO(f.read()))

X             = np.load(X_PATH)
y             = np.load(Y_PATH)
target_scaler = load_scaler(TARGET_SCALER)

print(f"  X shape : {X.shape}")
print(f"  y shape : {y.shape}")

y_binary  = (y.flatten() > 0).astype(np.float32)
rain_mask = y.flatten() > 0
X_rain    = X[rain_mask]
y_rain    = y[rain_mask]

print(f"  Dry  : {(y_binary==0).sum():,}  ({100*(y_binary==0).mean():.1f}%)")
print(f"  Rain : {(y_binary==1).sum():,}  ({100*(y_binary==1).mean():.1f}%)")
print(f"  Stage 2 rows : {len(X_rain):,}")


# ─────────────────────────────────────────────────────────────────
# CELL 4 — Chronological Splits
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  STEP 2: Chronological Splits")
print("="*55)

test_idx   = np.load(TEST_INDICES)
test_start = int(test_idx[0])
val_start  = int(test_start * (1 - VAL_SPLIT))

X_tr,  y1_tr  = X[:val_start],           y_binary[:val_start]
X_vl,  y1_vl  = X[val_start:test_start], y_binary[val_start:test_start]
X_te,  y1_te  = X[test_start:],          y_binary[test_start:]

n2          = len(X_rain)
ts2         = int(n2 * (1 - TEST_SPLIT))
vs2         = int(ts2 * (1 - VAL_SPLIT))
X2_tr, y2_tr = X_rain[:vs2],    y_rain[:vs2]
X2_vl, y2_vl = X_rain[vs2:ts2], y_rain[vs2:ts2]
X2_te, y2_te = X_rain[ts2:],    y_rain[ts2:]

print(f"  Stage 1 — Train:{len(X_tr):,} Val:{len(X_vl):,} Test:{len(X_te):,}")
print(f"  Stage 2 — Train:{len(X2_tr):,} Val:{len(X2_vl):,} Test:{len(X2_te):,}")


# ─────────────────────────────────────────────────────────────────
# CELL 5 — TFT Building Blocks (Keras-3 compatible)
# ─────────────────────────────────────────────────────────────────

# ── Positional Encoding as proper Keras Layer
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, time_steps, d_model, **kwargs):
        super().__init__(**kwargs)
        positions = np.arange(time_steps)[:, np.newaxis]
        dims      = np.arange(d_model)[np.newaxis, :]
        angles    = positions / np.power(10000, (2*(dims//2)) / d_model)
        angles[:, 0::2] = np.sin(angles[:, 0::2])
        angles[:, 1::2] = np.cos(angles[:, 1::2])
        # Store as non-trainable weight
        self.pe = tf.constant(angles[np.newaxis], dtype=tf.float32)

    def call(self, x):
        return x + self.pe

    def get_config(self):
        config = super().get_config()
        return config


# ── GRN as a proper Keras Layer (avoids KerasTensor issues)
class GatedResidualNetwork(tf.keras.layers.Layer):
    def __init__(self, units, dropout_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.units        = units
        self.dropout_rate = dropout_rate
        self.dense_elu    = Dense(units, activation='elu')
        self.dense_linear = Dense(units)
        self.dropout      = Dropout(dropout_rate)
        self.dense_gate   = Dense(units, activation='sigmoid')
        self.dense_proj   = None   # built on first call if needed
        self.layer_norm   = LayerNormalization()

    def build(self, input_shape):
        if input_shape[-1] != self.units:
            self.dense_proj = Dense(self.units)
        super().build(input_shape)

    def call(self, x, training=False):
        residual = x
        h = self.dense_elu(x)
        h = self.dense_linear(h)
        h = self.dropout(h, training=training)
        gate = self.dense_gate(h)
        h    = h * gate
        if self.dense_proj is not None:
            residual = self.dense_proj(residual)
        out = h + residual
        return self.layer_norm(out)

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'dropout_rate': self.dropout_rate})
        return config


# ── Transformer block as proper Keras Layer
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, n_heads, ff_dim, dropout_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.attn     = MultiHeadAttention(
            num_heads=n_heads,
            key_dim=d_model // n_heads,
            dropout=dropout_rate
        )
        self.drop1    = Dropout(dropout_rate)
        self.norm1    = LayerNormalization(epsilon=1e-6)
        self.ff1      = Dense(ff_dim, activation='gelu')
        self.drop2    = Dropout(dropout_rate)
        self.ff2      = Dense(d_model)
        self.drop3    = Dropout(dropout_rate)
        self.norm2    = LayerNormalization(epsilon=1e-6)

    def call(self, x, training=False):
        attn_out = self.attn(x, x, training=training)
        attn_out = self.drop1(attn_out, training=training)
        x        = self.norm1(x + attn_out)
        ff_out   = self.ff1(x)
        ff_out   = self.drop2(ff_out, training=training)
        ff_out   = self.ff2(ff_out)
        ff_out   = self.drop3(ff_out, training=training)
        x        = self.norm2(x + ff_out)
        return x

    def get_config(self):
        config = super().get_config()
        return config


# ── VSN as proper Keras Layer — fixes the KerasTensor error
class VariableSelectionNetwork(tf.keras.layers.Layer):
    """
    Each feature gets its own GRN pathway.
    A softmax gate then learns which features matter most.
    Fully implemented as a Keras Layer — no raw tf.ops on KerasTensors.
    """
    def __init__(self, n_features, d_model, dropout_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.n_features   = n_features
        self.d_model      = d_model
        self.dropout_rate = dropout_rate

        # One dense projection per feature
        self.feature_projs = [Dense(d_model) for _ in range(n_features)]
        # One GRN per feature
        self.feature_grns  = [
            GatedResidualNetwork(d_model, dropout_rate)
            for _ in range(n_features)
        ]
        # Selection pathway
        self.selection_proj = Dense(d_model)
        self.selection_grn  = GatedResidualNetwork(d_model, dropout_rate)
        self.selection_gate = Dense(n_features, activation='softmax')

    def call(self, x, training=False):
        # x shape: (batch, time, n_features)

        # Per-feature processing
        var_outs = []
        for i in range(self.n_features):
            # Slice one feature: (batch, time, 1)
            vi = x[:, :, i:i+1]
            vi = self.feature_projs[i](vi)
            vi = self.feature_grns[i](vi, training=training)
            var_outs.append(vi)  # each: (batch, time, d_model)

        # Stack along new axis: (batch, time, n_features, d_model)
        stacked = tf.stack(var_outs, axis=2)

        # Selection weights from the full input context
        ctx     = self.selection_proj(x)
        ctx     = self.selection_grn(ctx, training=training)
        weights = self.selection_gate(ctx)        # (batch, time, n_features)
        weights = tf.expand_dims(weights, axis=-1) # (batch, time, n_features, 1)

        # Weighted sum across features → (batch, time, d_model)
        selected = tf.reduce_sum(stacked * weights, axis=2)
        return selected

    def get_config(self):
        config = super().get_config()
        config.update({
            'n_features'  : self.n_features,
            'd_model'     : self.d_model,
            'dropout_rate': self.dropout_rate
        })
        return config


# ── Model builders
def build_tft_stage1(
    time_steps=TIME_STEPS, n_features=N_FEATURES,
    d_model=D_MODEL, n_heads=N_HEADS,
    ff_dim=FF_DIM, n_blocks=N_BLOCKS,
    dropout_rate=DROPOUT_RATE
):
    inputs = Input(shape=(time_steps, n_features))

    x = VariableSelectionNetwork(n_features, d_model, dropout_rate)(inputs)
    x = PositionalEncoding(time_steps, d_model)(x)
    x = LSTM(d_model, return_sequences=True)(x)
    x = Dropout(dropout_rate)(x)

    for _ in range(n_blocks):
        x = TransformerBlock(d_model, n_heads, ff_dim, dropout_rate)(x)

    x = GatedResidualNetwork(d_model, dropout_rate)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(32, activation='relu')(x)
    out = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, out, name='TFT_S1_Classifier')
    model.compile(
        optimizer=Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return model


def build_tft_stage2(
    time_steps=TIME_STEPS, n_features=N_FEATURES,
    d_model=D_MODEL, n_heads=N_HEADS,
    ff_dim=FF_DIM, n_blocks=N_BLOCKS,
    dropout_rate=DROPOUT_RATE
):
    inputs = Input(shape=(time_steps, n_features))

    x = VariableSelectionNetwork(n_features, d_model, dropout_rate)(inputs)
    x = PositionalEncoding(time_steps, d_model)(x)
    x = LSTM(d_model, return_sequences=True)(x)
    x = Dropout(dropout_rate)(x)

    for _ in range(n_blocks):
        x = TransformerBlock(d_model, n_heads, ff_dim, dropout_rate)(x)

    x = GatedResidualNetwork(d_model, dropout_rate)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(32, activation='relu')(x)
    out = Dense(1, activation='linear')(x)

    model = Model(inputs, out, name='TFT_S2_Regressor')
    model.compile(
        optimizer=Adam(1e-3),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae']
    )
    return model


CUSTOM_OBJECTS = {
    'PositionalEncoding'      : PositionalEncoding,
    'GatedResidualNetwork'    : GatedResidualNetwork,
    'TransformerBlock'        : TransformerBlock,
    'VariableSelectionNetwork': VariableSelectionNetwork,
}


def get_callbacks(save_path, patience=10):
    return [
        EarlyStopping(monitor='val_loss', patience=patience,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint(save_path, monitor='val_loss',
                        save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=5, min_lr=1e-6, verbose=1)
    ]


# ─────────────────────────────────────────────────────────────────
# CELL 6 — Train TFT Stage 1
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  STAGE 1 — TFT Rain/No-Rain Classifier")
print("="*55)

rain_ratio   = y_binary.mean()
class_weight = {0: 1.0, 1: round(float((1 - rain_ratio) / rain_ratio), 4)}
print(f"  Class weights: {class_weight}")

tf.keras.backend.clear_session()
tft_s1 = build_tft_stage1()
tft_s1.summary()

history_s1 = tft_s1.fit(
    X_tr, y1_tr,
    validation_data=(X_vl, y1_vl),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=get_callbacks(TFT_S1_PATH, patience=10),
    verbose=1
)

print(f"\n  TFT Stage 1 Test Evaluation:")
y1_prob = tft_s1.predict(X_te, batch_size=BATCH_SIZE, verbose=0).flatten()
y1_pred = (y1_prob > 0.5).astype(int)
print(classification_report(y1_te, y1_pred, target_names=['Dry', 'Rain']))
print(f"  Confusion Matrix:\n{confusion_matrix(y1_te, y1_pred)}")

plt.figure(figsize=(12, 4))
plt.subplot(1,2,1)
plt.plot(history_s1.history['loss'],     label='Train')
plt.plot(history_s1.history['val_loss'], label='Val')
plt.title('TFT S1 — Loss'); plt.legend()
plt.subplot(1,2,2)
plt.plot(history_s1.history['accuracy'],     label='Train')
plt.plot(history_s1.history['val_accuracy'], label='Val')
plt.title('TFT S1 — Accuracy'); plt.legend()
plt.tight_layout()
plt.savefig('tft_stage1_curves.png', dpi=150)
plt.show()
print(f"  Saved → {TFT_S1_PATH}")


# ─────────────────────────────────────────────────────────────────
# CELL 7 — Train TFT Stage 2
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  STAGE 2 — TFT Rainfall Amount Regressor")
print("="*55)
print(f"  Training on {len(X2_tr):,} rainy samples only")

tft_s2 = build_tft_stage2()
tft_s2.summary()

history_s2 = tft_s2.fit(
    X2_tr, y2_tr,
    validation_data=(X2_vl, y2_vl),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=get_callbacks(TFT_S2_PATH, patience=10),
    verbose=1
)

y2_pred_s  = tft_s2.predict(X2_te, batch_size=BATCH_SIZE, verbose=0)
y2_pred_mm = target_scaler.inverse_transform(np.clip(y2_pred_s, 0, 1))
y2_true_mm = target_scaler.inverse_transform(y2_te)
mae2  = mean_absolute_error(y2_true_mm, y2_pred_mm)
rmse2 = np.sqrt(mean_squared_error(y2_true_mm, y2_pred_mm))
print(f"\n  TFT Stage 2 MAE  : {mae2:.4f} mm/hr")
print(f"  TFT Stage 2 RMSE : {rmse2:.4f} mm/hr")

plt.figure(figsize=(12, 4))
plt.subplot(1,2,1)
plt.plot(history_s2.history['loss'],     label='Train')
plt.plot(history_s2.history['val_loss'], label='Val')
plt.title('TFT S2 — Loss'); plt.legend()
plt.subplot(1,2,2)
plt.plot(history_s2.history['mae'],     label='Train')
plt.plot(history_s2.history['val_mae'], label='Val')
plt.title('TFT S2 — MAE'); plt.legend()
plt.tight_layout()
plt.savefig('tft_stage2_curves.png', dpi=150)
plt.show()
print(f"  Saved → {TFT_S2_PATH}")


# ─────────────────────────────────────────────────────────────────
# CELL 8 — Combined TFT Inference on Test Set
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  COMBINED TFT TWO-STAGE INFERENCE")
print("="*55)

prob_rain    = tft_s1.predict(X_te, batch_size=BATCH_SIZE, verbose=0).flatten()
rain_flags   = prob_rain > 0.5

final_scaled = np.zeros((len(X_te), 1), dtype=np.float32)
if rain_flags.sum() > 0:
    final_scaled[rain_flags] = tft_s2.predict(
        X_te[rain_flags], batch_size=BATCH_SIZE, verbose=0
    )

final_scaled  = np.clip(final_scaled, 0, 1)
final_pred_mm = np.clip(
    target_scaler.inverse_transform(final_scaled), 0, None
).flatten()
final_true_mm = np.clip(
    target_scaler.inverse_transform(y[test_start:]), 0, None
).flatten()

mae_tft  = mean_absolute_error(final_true_mm, final_pred_mm)
rmse_tft = np.sqrt(mean_squared_error(final_true_mm, final_pred_mm))
print(f"  TFT Combined MAE  : {mae_tft:.4f} mm/hr")
print(f"  TFT Combined RMSE : {rmse_tft:.4f} mm/hr")
print(f"\n  CNN-LSTM MAE  : 0.1331 mm/hr")
print(f"  TFT      MAE  : {mae_tft:.4f} mm/hr")
diff = 0.1331 - mae_tft
print(f"  Difference    : {diff:+.4f}  ({'TFT better ✓' if diff > 0 else 'CNN-LSTM better'})")

plt.figure(figsize=(14, 4))
plt.plot(final_true_mm[:500], label='Actual',   alpha=0.75)
plt.plot(final_pred_mm[:500], label='TFT Pred', alpha=0.75)
plt.title('Two-Stage TFT — Predictions vs Actual (mm/hr)')
plt.xlabel('Sample Hour')
plt.ylabel('Precipitation (mm/hr)')
plt.legend()
plt.tight_layout()
plt.savefig('tft_predictions_v3.png', dpi=150)
plt.show()


# ─────────────────────────────────────────────────────────────────
# CELL 9 — Save TFT Predictions for Ensemble
# ─────────────────────────────────────────────────────────────────
np.save('tft_test_pred_scaled.npy', final_scaled)
print(f"\n  Saved → tft_test_pred_scaled.npy  shape: {final_scaled.shape}")


# ─────────────────────────────────────────────────────────────────
# CELL 10 — Final Summary
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  TRAINING COMPLETE — SCRIPT 2 v3")
print("="*55)
print(f"""
  Models saved:
    {TFT_S1_PATH}
    {TFT_S2_PATH}

  TFT performance (mm/hr):
    Combined MAE  : {mae_tft:.4f}
    Combined RMSE : {rmse_tft:.4f}

  CNN-LSTM (Script 1):
    Combined MAE  : 0.1331
    Combined RMSE : 0.3810

  Next: Run Script 3 — train_ensemble.py (fresh session)
  Files needed:
    stage1_classifier.keras
    stage2_regressor.keras
    tft_stage1_classifier.keras
    tft_stage2_regressor.keras
    cnn_lstm_test_pred_scaled.npy
    tft_test_pred_scaled.npy
    test_indices.npy
""")

Mounting Google Drive...
Mounted at /content/drive
Working directory: /content/drive/MyDrive/GNSS_csir
GPU detected: /physical_device:GPU:0

  STEP 1: Loading Data
  X shape : (134662, 48, 13)
  y shape : (134662, 1)
  Dry  : 65,154  (48.4%)
  Rain : 69,508  (51.6%)
  Stage 2 rows : 69,508

  STEP 2: Chronological Splits
  Stage 1 — Train:103,015 Val:18,180 Test:13,467
  Stage 2 — Train:53,173 Val:9,384 Test:6,951

  STAGE 1 — TFT Rain/No-Rain Classifier
  Class weights: {0: 1.0, 1: 0.9374}


Model: "TFT_S1_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 48, 13)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ variable_selection_network      │ (None, 48, 128)        │       703,885 │
│ (VariableSelectionNetwork)      │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding             │ (None, 48, 128)        │             0 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 48, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 48, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 48, 128)        │       132,480 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 48, 128)        │       132,480 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, 48, 128)        │       132,480 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ (None, 48, 128)        │       132,480 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gated_residual_network_14       │ (None, 48, 128)        │        49,792 │
│ (GatedResidualNetwork)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_68 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_32 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_69 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_70 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,425,550 (5.44 MB)

 Trainable params: 1,425,550 (5.44 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
403/403 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.7980 - auc: 0.8639 - loss: 0.4220 - precision: 0.7771 - recall: 0.8309
Epoch 1: val_loss improved from None to 0.32574, saving model to tft_stage1_classifier.keras

Epoch 1: finished saving model to tft_stage1_classifier.keras
403/403 ━━━━━━━━━━━━━━━━━━━━ 122s 191ms/step - accuracy: 0.8354 - auc: 0.9087 - loss: 0.3613 - precision: 0.8135 - recall: 0.8747 - val_accuracy: 0.8536 - val_auc: 0.9348 - val_loss: 0.3257 - val_precision: 0.8183 - val_recall: 0.9272 - learning_rate: 0.0010
Epoch 2/100
403/403 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step - accuracy: 0.8592 - auc: 0.9375 - loss: 0.3075 - precision: 0.8564 - recall: 0.8651
Epoch 2: val_loss did not improve from 0.32574
403/403 ━━━━━━━━━━━━━━━━━━━━ 73s 182ms/step - accuracy: 0.8649 - auc: 0.9415 - loss: 0.2977 - precision: 0.8638 - recall: 0.8698 - val_accuracy: 0.8504 - val_auc: 0.9269 - val_loss: 0.3430 - val_precision: 0.8169 - val_recall: 0.9219 - learning_rate: 0.0010


In [ ]:
import tensorflow as tf
import numpy as np
from google.colab import drive
drive.mount('/content/drive')


# Redefine PositionalEncoding with a from_config that handles missing args
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, time_steps=48, d_model=128, **kwargs):
        super().__init__(**kwargs)
        self.time_steps = time_steps
        self.d_model    = d_model
        positions = np.arange(time_steps)[:, np.newaxis]
        dims      = np.arange(d_model)[np.newaxis, :]
        angles    = positions / np.power(10000, (2*(dims//2)) / d_model)
        angles[:, 0::2] = np.sin(angles[:, 0::2])
        angles[:, 1::2] = np.cos(angles[:, 1::2])
        self.pe = tf.constant(angles[np.newaxis], dtype=tf.float32)

    def call(self, x):
        return x + self.pe

    def get_config(self):
        config = super().get_config()
        config.update({'time_steps': self.time_steps, 'd_model': self.d_model})
        return config

    @classmethod
    def from_config(cls, config):
        # Handle old saved models that didn't store time_steps/d_model
        config.setdefault('time_steps', 48)
        config.setdefault('d_model', 128)
        return cls(**config)


class GatedResidualNetwork(tf.keras.layers.Layer):
    def __init__(self, units, dropout_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.units        = units
        self.dropout_rate = dropout_rate
        self.dense_elu    = tf.keras.layers.Dense(units, activation='elu')
        self.dense_linear = tf.keras.layers.Dense(units)
        self.dropout      = tf.keras.layers.Dropout(dropout_rate)
        self.dense_gate   = tf.keras.layers.Dense(units, activation='sigmoid')
        self.dense_proj   = None
        self.layer_norm   = tf.keras.layers.LayerNormalization()

    def build(self, input_shape):
        if input_shape[-1] != self.units:
            self.dense_proj = tf.keras.layers.Dense(self.units)
        super().build(input_shape)

    def call(self, x, training=False):
        residual = x
        h = self.dense_elu(x)
        h = self.dense_linear(h)
        h = self.dropout(h, training=training)
        gate = self.dense_gate(h)
        h    = h * gate
        if self.dense_proj is not None:
            residual = self.dense_proj(residual)
        return self.layer_norm(h + residual)

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'dropout_rate': self.dropout_rate})
        return config


class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model=128, n_heads=8, ff_dim=256,
                 dropout_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.d_model      = d_model
        self.n_heads      = n_heads
        self.ff_dim       = ff_dim
        self.dropout_rate = dropout_rate
        self.attn  = tf.keras.layers.MultiHeadAttention(
            num_heads=n_heads, key_dim=d_model//n_heads, dropout=dropout_rate)
        self.drop1 = tf.keras.layers.Dropout(dropout_rate)
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ff1   = tf.keras.layers.Dense(ff_dim, activation='gelu')
        self.drop2 = tf.keras.layers.Dropout(dropout_rate)
        self.ff2   = tf.keras.layers.Dense(d_model)
        self.drop3 = tf.keras.layers.Dropout(dropout_rate)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, x, training=False):
        attn = self.attn(x, x, training=training)
        attn = self.drop1(attn, training=training)
        x    = self.norm1(x + attn)
        ff   = self.ff1(x)
        ff   = self.drop2(ff, training=training)
        ff   = self.ff2(ff)
        ff   = self.drop3(ff, training=training)
        return self.norm2(x + ff)

    def get_config(self):
        config = super().get_config()
        config.update({
            'd_model': self.d_model, 'n_heads': self.n_heads,
            'ff_dim': self.ff_dim, 'dropout_rate': self.dropout_rate
        })
        return config

    @classmethod
    def from_config(cls, config):
        config.setdefault('d_model', 128)
        config.setdefault('n_heads', 8)
        config.setdefault('ff_dim', 256)
        config.setdefault('dropout_rate', 0.2)
        return cls(**config)


class VariableSelectionNetwork(tf.keras.layers.Layer):
    def __init__(self, n_features, d_model, dropout_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.n_features    = n_features
        self.d_model       = d_model
        self.dropout_rate  = dropout_rate
        self.feature_projs = [tf.keras.layers.Dense(d_model)
                               for _ in range(n_features)]
        self.feature_grns  = [GatedResidualNetwork(d_model, dropout_rate)
                               for _ in range(n_features)]
        self.selection_proj = tf.keras.layers.Dense(d_model)
        self.selection_grn  = GatedResidualNetwork(d_model, dropout_rate)
        self.selection_gate = tf.keras.layers.Dense(n_features,
                                                     activation='softmax')

    def call(self, x, training=False):
        var_outs = []
        for i in range(self.n_features):
            vi = x[:, :, i:i+1]
            vi = self.feature_projs[i](vi)
            vi = self.feature_grns[i](vi, training=training)
            var_outs.append(vi)
        stacked = tf.stack(var_outs, axis=2)
        ctx     = self.selection_proj(x)
        ctx     = self.selection_grn(ctx, training=training)
        weights = self.selection_gate(ctx)
        weights = tf.expand_dims(weights, axis=-1)
        return tf.reduce_sum(stacked * weights, axis=2)

    def get_config(self):
        config = super().get_config()
        config.update({
            'n_features': self.n_features,
            'd_model': self.d_model,
            'dropout_rate': self.dropout_rate
        })
        return config


CUSTOM_OBJECTS = {
    'PositionalEncoding'      : PositionalEncoding,
    'GatedResidualNetwork'    : GatedResidualNetwork,
    'TransformerBlock'        : TransformerBlock,
    'VariableSelectionNetwork': VariableSelectionNetwork,
}

# Try loading the old TFT Stage 1
tft_s1 = tf.keras.models.load_model(
    'your_path/tft_stage1_classifier.keras',
    custom_objects=CUSTOM_OBJECTS
)
print("TFT Stage 1 loaded successfully ✓")
print(f"Params: {tft_s1.count_params():,}")

# Quick test
import numpy as np
X_test_sample = np.random.randn(5, 48, 13).astype(np.float32)
out = tft_s1.predict(X_test_sample, verbose=0)
print(f"Test output: {out.flatten()}")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'variable_selection_network', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'transformer_block', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'transformer_block_1'

TFT Stage 1 loaded successfully ✓
Params: 1,425,550
Test output: [0.9681652  0.00400906 0.9195542  0.15484485 0.8851051 ]
